# Causal Survival Analysis with PyTorch: Deep Causal Cox Model (TARNet + Cox Partial Likelihood)

This notebook implements a **Two-Head Causal Survival Neural Network** combining the **TARNet** representation learning architecture with a **Cox Proportional Hazards Partial Likelihood Loss** to properly handle **right-censored time-to-event data**.

### Dataset: ACTG 320 Clinical Trial (`sksurv.datasets.load_aids`)
- **Treatment ($T$):** `tx` (1 = Indinavir + ZDV/d4T + 3TC triple therapy; 0 = ZDV/d4T + 3TC dual therapy)
- **Outcome ($Y, E$):** Time to AIDS-defining event or death (`time`) and censoring indicator (`censor`)
- **Objective:** Estimate **Individualized Treatment Effects (ITE)** in terms of **Log-Hazard Ratio Difference** ($\Delta \hat{h}(x) = \hat{h}_1(x) - \hat{h}_0(x)$), where negative values indicate reduced hazard (protective treatment effect).

In [ ]:
# Install required survival analysis package
!pip install -q scikit-survival

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sksurv.datasets import load_aids

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

## 1. Load & Preprocess ACTG 320 Clinical Trial Data
We extract the actual randomized treatment variable (`tx`) and encode baseline covariates cleanly without data leakage.

In [ ]:
# Load ACTG 320 dataset
data_x, data_y = load_aids()

# Inspect columns to verify actual treatment variable
print("Available columns:", list(data_x.columns))

# Extract actual randomized treatment indicator ('tx': 1 = IDV group, 0 = Control group)
if 'tx' in data_x.columns:
    T = data_x['tx'].astype(int).values
    X_df = data_x.drop(columns=['tx']).copy()
else:
    raise KeyError("Treatment column 'tx' not found in dataset.")

# Encode categorical/ordinal baseline covariates
for col in X_df.select_dtypes(include=['category', 'object']).columns:
    X_df[col] = X_df[col].astype('category').cat.codes

# Extract censoring indicator (event = True/1) and time-to-event
event_indicator = np.array([int(row[0]) for row in data_y], dtype=np.float32)
event_time = np.array([float(row[1]) for row in data_y], dtype=np.float32)

# Standardize covariates safely
X_arr = np.array(X_df.values, dtype=np.float64)
X_mean = X_arr.mean(axis=0)
X_std = X_arr.std(axis=0) + 1e-8
X_scaled = ((X_arr - X_mean) / X_std).astype(np.float32)

print(f"Sample size (N): {X_scaled.shape[0]}, Covariates (P): {X_scaled.shape[1]}")
print(f"Treatment Arm -> Control (T=0): {np.sum(T==0)}, Treated (T=1): {np.sum(T==1)}")
print(f"Observed Event Rate: {event_indicator.mean():.2%} (Censored: {1 - event_indicator.mean():.2%})")

## 2. Two-Head Causal Survival Architecture (TARNet) & Cox Partial Likelihood Loss
Instead of naive MSE regression (which ignores right-censoring), we optimize the **Negative Log Cox Partial Likelihood**:
$$\mathcal{L}_{\text{Cox}}(\theta) = - \sum_{i: E_i = 1} \left( \hat{h}_{T_i}(x_i) - \log \sum_{j \in \mathcal{R}(Y_i)} \exp(\hat{h}_{T_j}(x_j)) \right)$$
where $\mathcal{R}(Y_i) = \{j : Y_j \ge Y_i\}$ is the risk set at time $Y_i$, and $\hat{h}_t(x)$ is the predicted log-risk under treatment $t \in \{0, 1\}$.

In [ ]:
class CausalCoxNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, dropout=0.2):
        super(CausalCoxNet, self).__init__()
        # Shared representation network phi(x)
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ELU()
        )
        # Potential outcome log-hazard head for Control (T=0)
        self.head_0 = nn.Sequential(
            nn.Linear(hidden_dim // 2, 8),
            nn.ELU(),
            nn.Linear(8, 1)
        )
        # Potential outcome log-hazard head for Treated (T=1)
        self.head_1 = nn.Sequential(
            nn.Linear(hidden_dim // 2, 8),
            nn.ELU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        phi = self.shared(x)
        h0 = self.head_0(phi)
        h1 = self.head_1(phi)
        return h0, h1

def cox_partial_likelihood_loss(log_hazard, durations, events):
    """
    Computes negative log Cox partial likelihood handling right-censored observations.
    """
    # Sort by descending survival duration so risk sets are cumulative sums
    order = torch.argsort(durations, descending=True)
    log_h = log_hazard[order].squeeze()
    e = events[order].squeeze()

    # Log-sum-exp trick for numerical stability over risk sets
    max_log_h = torch.max(log_h)
    exp_log_h = torch.exp(log_h - max_log_h)
    log_risk_set = max_log_h + torch.log(torch.cumsum(exp_log_h, dim=0) + 1e-8)

    # Sum over uncensored events only
    uncensored_likelihood = log_h - log_risk_set
    num_events = torch.sum(e) + 1e-8
    loss = -torch.sum(uncensored_likelihood * e) / num_events
    return loss

## 3. Model Training with Weight Decay Regularization

In [ ]:
X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)
t_tensor = torch.tensor(T, dtype=torch.float32).unsqueeze(1).to(device)
y_tensor = torch.tensor(event_time, dtype=torch.float32).to(device)
e_tensor = torch.tensor(event_indicator, dtype=torch.float32).to(device)

model = CausalCoxNet(input_dim=X_scaled.shape[1], hidden_dim=32, dropout=0.2).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)

loss_history = []
model.train()
for epoch in range(150):
    optimizer.zero_grad()
    h0_pred, h1_pred = model(X_tensor)
    
    # Factual log-hazard prediction based on observed treatment assignment
    h_factual = t_tensor * h1_pred + (1.0 - t_tensor) * h0_pred
    loss = cox_partial_likelihood_loss(h_factual, y_tensor, e_tensor)
    
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch [{epoch+1}/150] | Cox Partial NLL: {loss.item():.4f}")

## 4. Estimate Individualized Treatment Effects (Log-Hazard Ratio Difference) & Export

In [ ]:
model.eval()
with torch.no_grad():
    h0_all, h1_all = model(X_tensor)
    # Individualized Log-Hazard Ratio: h1(x) - h0(x)
    # Negative values (< 0) indicate treatment reduces hazard (beneficial)
    ite_log_hr = (h1_all - h0_all).cpu().numpy().flatten()
    ite_hr = np.exp(ite_log_hr)

results_df = pd.DataFrame({
    'Observed_Treatment_tx': T,
    'Duration_Days': event_time,
    'Event_Observed': event_indicator.astype(int),
    'Pred_LogHazard_Control_h0': h0_all.cpu().numpy().flatten(),
    'Pred_LogHazard_Treated_h1': h1_all.cpu().numpy().flatten(),
    'ITE_Log_Hazard_Ratio': ite_log_hr,
    'ITE_Hazard_Ratio': ite_hr
})

results_df.to_csv('actg320_causal_cox_ite.csv', index=False)
print(f"Average Treatment Effect on Log-HR Scale (ATE): {ite_log_hr.mean():.4f} (Mean HR: {np.exp(ite_log_hr.mean()):.4f})")
print(f"Proportion of patients benefiting from treatment (Log-HR < 0): {(ite_log_hr < 0).mean():.2%}")
display(results_df.head(10))

## 5. Diagnostic & Heterogeneity Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Plot 1: Training Convergence
axes[0].plot(loss_history, color='#2b5c8f', linewidth=2)
axes[0].set_title('Cox Partial Likelihood Convergence')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Negative Log Partial Likelihood')
axes[0].grid(alpha=0.3)

# Plot 2: Distribution of Individualized Log-Hazard Ratios
sns.histplot(results_df['ITE_Log_Hazard_Ratio'], kde=True, color='#4a7c59', ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Null Effect (Log-HR = 0)')
axes[1].axvline(ite_log_hr.mean(), color='black', linestyle=':', linewidth=1.5, label=f'Mean ATE = {ite_log_hr.mean():.2f}')
axes[1].set_title('Distribution of Individualized Log-Hazard Ratios (ITE)')
axes[1].set_xlabel('Estimated Log-Hazard Ratio ($\hat{h}_1(x) - \hat{h}_0(x)$)')
axes[1].set_ylabel('Patient Count')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()